# Download Trained Model

This notebook downloads the fine-tuned model from Azure ML after training completes.

## What This Notebook Does

1. **Lists Training Jobs**: Shows recent training runs with status and metrics
2. **Validates Job Completion**: Ensures selected job finished successfully
3. **Downloads Model Artifacts**: Retrieves trained model weights, tokenizer, and metadata
4. **Verifies Model Integrity**: Checks downloaded files are complete and loadable
5. **Prepares for Optimization**: Organizes model files for next pipeline stage

## Why Download the Model?

**Local Optimization:**
- Quantization and ONNX export run faster locally than on Azure ML
- No compute charges while performing optimization experiments
- Easier debugging with local Python debugger

**Model Archiving:**
- Keep local copies of trained models for comparison
- Version control model checkpoints alongside code
- Share models with team via storage or model registry

**Next Pipeline Stage:**
- Optimization notebook (08) requires local model access
- Container build needs model files in `models/` directory
- Inference testing runs on local hardware

## Download Process

### Step 1: Find Training Job
- Lists jobs from experiment (e.g., `phi-4-training`)
- Shows job status, duration, and final metrics
- User selects job by ID or name

### Step 2: Locate Outputs
Azure ML training jobs produce:
- **Model checkpoint**: `outputs/checkpoint-final/` or `outputs/model/`
- **Tokenizer files**: `vocab.json`, `tokenizer_config.json`, `special_tokens_map.json`
- **Training logs**: `logs/`, `metrics.json`
- **Config files**: `config.json`, `training_args.json`

### Step 3: Download Files
- Uses Azure ML SDK to download from job outputs
- Preserves directory structure
- Validates file sizes and checksums

### Step 4: Organize Locally
Downloaded to `models/trained/{model_name}/`:
```
models/trained/phi-4-finetuned/
├── config.json
├── model.safetensors (or pytorch_model.bin)
├── tokenizer.json
├── tokenizer_config.json
├── special_tokens_map.json
└── training_args.json
```

## What If Job Is Still Running?

- Notebook shows job status and progress
- You can wait for completion or cancel and retry later
- Partial checkpoints can be downloaded for early stopping analysis

## Model Size Expectations

| Model Type | Typical Size | Download Time |
|------------|--------------|---------------|
| Phi-3 Mini (3.8B) | 7-8GB | 5-10 min |
| Phi-4 Mini (14B) | 14-16GB | 10-20 min |
| Llama-3-8B | 16-18GB | 15-25 min |

## Prerequisites

- Completed notebook `06-submit-training-job.ipynb`
- Training job completed successfully (status: "Completed")
- Sufficient local disk space (~20GB for model + checkpoints)
- Azure ML workspace credentials valid

## Expected Duration

~10-30 minutes depending on model size and network speed

## 1. Setup and Connect to Workspace

**Why use ModelDownloader?** The `ModelDownloader` class handles Azure ML SDK complexity for locating job outputs, downloading with progress tracking, validating checksums, and organizing files in the correct directory structure.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from src.training.model_downloader import ModelDownloader
import pandas as pd

# Initialize Azure ML client
credential = DefaultAzureCredential()
ml_client = MLClient.from_config(credential=credential)

print(f"✓ Connected to workspace: {ml_client.workspace_name}")

# Initialize model downloader
downloader = ModelDownloader(ml_client=ml_client)
print("✓ Model downloader initialized")

## 2. Find Your Training Job

**Why list jobs?** You need the job ID to download its outputs. This table shows recent training runs so you can select the correct one, especially if you've run multiple experiments with different hyperparameters.

In [ ]:
# List recent training jobs
experiment_name = "phi-4-training"  # Update if different

jobs = ml_client.jobs.list(max_results=10)
job_list = []

for job in jobs:
    if job.experiment_name == experiment_name:
        job_list.append({
            "Job ID": job.name,
            "Display Name": job.display_name,
            "Status": job.status,
            "Created": job.creation_context.created_at.strftime("%Y-%m-%d %H:%M"),
        })

if job_list:
    df = pd.DataFrame(job_list)
    print(f"Recent training jobs in '{experiment_name}':")
    print(df.to_string(index=False))
else:
    print(f"No jobs found in experiment: {experiment_name}")

In [ ]:
# Set your job ID here (from the table above)
job_name = "<paste-job-id-here>"  # e.g., "kind_drum_abc123"

# Verify job exists and is complete
try:
    job = ml_client.jobs.get(job_name)
    print(f"Job: {job.name}")
    print(f"Status: {job.status}")
    print(f"Studio URL: {job.studio_url}")

    if job.status != "Completed":
        print(f"\n⚠️  Warning: Job status is '{job.status}', not 'Completed'")
        print("Model download may fail or be incomplete.")
except Exception as e:
    print(f"❌ Job not found: {e}")
    print("Please check the job ID and try again.")

## 3. List Available Checkpoints

In [ ]:
# List checkpoints saved during training
print("Listing checkpoints...")
checkpoints = downloader.list_job_checkpoints(job_name)

if checkpoints:
    print(f"\nFound {len(checkpoints)} checkpoints:")
    for i, checkpoint in enumerate(checkpoints, 1):
        print(f"  {i}. {checkpoint}")
else:
    print("⚠️  No checkpoints found in job outputs")

In [ ]:
# Identify best checkpoint
best_checkpoint = downloader.get_best_checkpoint(job_name)
print(f"\n✓ Best checkpoint identified: {best_checkpoint}")

## 4. Download Model Checkpoint

In [ ]:
# Configure download
output_path = project_root / "models" / "trained" / job_name
checkpoint_to_download = best_checkpoint  # Or choose specific checkpoint

print(f"Downloading checkpoint: {checkpoint_to_download}")
print(f"Output path: {output_path}")
print("\nThis may take several minutes for large models...\n")

# Download model
model_path = downloader.download_from_job(
    job_name=job_name,
    output_path=str(output_path),
    checkpoint=checkpoint_to_download,
)

print(f"\n✓ Model downloaded to: {model_path}")

## 5. Validate Model Files

In [ ]:
# Validate downloaded files
validation_results = downloader.validate_model_files(model_path)

print("\nModel File Validation:")
print("=" * 50)

all_valid = True
for file, present in validation_results.items():
    status = "✓ Present" if present else "✗ Missing"
    print(f"{status:12} - {file}")
    if not present:
        all_valid = False

print("=" * 50)
if all_valid:
    print("✓ All required files present")
else:
    print("⚠️  Some files are missing - model may not load correctly")

## 6. Extract Model Metadata

In [ ]:
# Extract and display metadata
metadata = downloader.extract_model_metadata(model_path)

print("\nModel Metadata:")
print("=" * 50)
print(f"Path: {metadata['path']}")
print(f"Size: {metadata['size_mb']:.1f} MB")
print(f"Model Type: {metadata['model_type']}")
print(f"Vocabulary Size: {metadata['vocab_size']:,}")
print(f"Hidden Size: {metadata['hidden_size']:,}")
print(f"Number of Layers: {metadata['num_layers']}")
print("=" * 50)

In [ ]:
# List all files in model directory
print("\nModel Files:")
for file_path in sorted(model_path.rglob("*")):
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        rel_path = file_path.relative_to(model_path)
        print(f"  {rel_path} ({size_mb:.1f} MB)")

## 7. Test Model Loading

In [ ]:
# Test loading the model with transformers
print("Testing model loading...")

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    print(f"✓ Tokenizer loaded: {len(tokenizer)} tokens")

    print("\nLoading model...")
    print("Note: This may take a few minutes for large models")

    # For LoRA models, load with PEFT
    adapter_config = model_path / "adapter_config.json"
    if adapter_config.exists():
        print("Detected LoRA adapter - loading with PEFT")
        from peft import PeftModel, PeftConfig

        # Need to load base model first
        print("⚠️  LoRA model requires base model for loading")
        print("   Base model path needed for inference")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            device_map="cpu",  # Use CPU for testing
            low_cpu_mem_usage=True,
        )
        print(f"✓ Model loaded successfully")
        print(f"   Parameters: {model.num_parameters():,}")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("\nModel files may be incomplete or corrupted.")

## 8. Optional: Register in Azure ML Model Registry

In [ ]:
# Register model in Azure ML for easy versioning and deployment
# Uncomment to register

# from azure.ai.ml.entities import Model

# model_name = "phi-4-finetuned"

# print(f"Registering model: {model_name}")

# model = Model(
#     name=model_name,
#     path=f"azureml://jobs/{job_name}/outputs/{checkpoint_to_download}",
#     description="Phi-4 fine-tuned with LoRA",
#     tags={
#         "framework": "pytorch",
#         "task": "text-generation",
#         "base_model": "phi-4",
#         "training_method": "lora",
#         "job_id": job_name,
#     },
# )

# registered_model = ml_client.models.create_or_update(model)

# print(f"\n✓ Model registered!")
# print(f"  Name: {registered_model.name}")
# print(f"  Version: {registered_model.version}")
# print(f"  ID: {registered_model.id}")

## Next Steps

Now that you have the trained model:
1. **Evaluate Model** (notebook 08): Run evaluation on test set
2. **Optimize Model** (notebook 09): Apply quantization and optimization
3. **Build Container** (notebook 10): Package for deployment

## Troubleshooting

### No Checkpoints Found
- Verify training job completed successfully
- Check job outputs in Azure ML Studio
- Ensure checkpointing was enabled in training config

### Model Files Missing
- Training may have failed partway through
- Check training logs for errors
- Try downloading different checkpoint

### Download Very Slow
- Large models (>10GB) can take 10+ minutes
- Check network connection
- Consider using Azure Storage Explorer for large files

### LoRA Model Won't Load
- LoRA adapters need base model
- Download base model first (notebook 04)
- Load base model, then apply LoRA adapter